In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

# 4:22:03 - 4:24:59 Workflow Intro & Setup
torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

# 4:32:23 Dataset Creation
weight = 0.7
bias = 0.3

X = torch.arange(-1, 1, 0.1).unsqueeze(dim=1)
y = weight * X + bias + 0.1 * torch.randn(X.size())

# 4:42:14 Data Splitting (80/20 split)
train_split = int(0.8 * len(X))
X_train, y_train = X[:train_split], y[:train_split]
X_test, y_test = X[train_split:], y[train_split:]

# 4:50:44 Data Visualization
def plot_predictions(train_data=X_train, train_labels=y_train, test_data=X_test, test_labels=y_test, predictions=None):
    plt.figure(figsize=(10, 7))
    
    plt.scatter(train_data, train_labels, c="b", s=4, label="Training data")
    plt.scatter(test_data, test_labels, c="g", s=4, label="Testing data")
    
    if predictions is not None:
        plt.scatter(test_data, predictions, c="r", s=4, label="Predictions")
    
    plt.legend(prop={"size": 14})
    plt.show()

plot_predictions()

# 4:58:39 - 5:12:59 Linear Model
class LinearRegressionModelV2(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear_layer = nn.Linear(in_features=1, out_features=1)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear_layer(x)

model_1 = LinearRegressionModelV2().to(device)
print(f"Model structure: {model_1}")
print(f"\nModel on device: {next(model_1.parameters()).device}")

# 5:19:19 Key PyTorch Classes & 5:25:55 Inspect Model
print(f"\nModel parameters: {list(model_1.parameters())}")
print(f"\nModel state_dict: {model_1.state_dict()}")

# 5:35:56 Predictions (untrained)
with torch.inference_mode():
    X_test_pred = model_1(X_test.to(device))
    
plot_predictions(predictions=X_test_pred.cpu())

# 5:47:18 Training Intuition & 5:55:43 Optimizer Setup
loss_fn = nn.L1Loss()
optimizer = torch.optim.SGD(model_1.parameters(), lr=0.01)

# 6:08:44 - 6:22:48 Training Loop
torch.manual_seed(42)
epochs = 200

for epoch in range(epochs):
    # 6:31:45 Training Steps Review
    # 1. Forward pass
    model_1.train()
    y_pred = model_1(X_train.to(device))
    
    # 2. Calculate loss (how wrong are we?)
    loss = loss_fn(y_pred, y_train.to(device))
    
    # 3. Optimizer zero grad (reset gradients)
    optimizer.zero_grad()
    
    # 4. Perform backpropagation (loss.backward() calculates gradients)
    loss.backward()
    
    # 5. Optimizer step (update parameters)
    optimizer.step()
    
    # Testing
    model_1.eval()
    with torch.inference_mode():
        test_pred = model_1(X_test.to(device))
        test_loss = loss_fn(test_pred, y_test.to(device))
        
    if epoch % 10 == 0:
        print(f"Epoch: {epoch} | Loss: {loss.item():.3f} | Test loss: {test_loss.item():.3f}")

# 6:46:52 Run Training Loop - Check results
print(f"\nFinal model parameters:")
print(f"Weight: {model_1.state_dict()['linear_layer.weight'].item():.3f}")
print(f"Expected weight: {weight}")
print(f"Bias: {model_1.state_dict()['linear_layer.bias'].item():.3f}")
print(f"Expected bias: {bias}")

# 6:56:28 - 7:08:15 Testing Code & Steps Review
with torch.inference_mode():
    y_preds = model_1(X_test.to(device))

plot_predictions(predictions=y_preds.cpu())

# 7:23:07 Model Save/Load
MODEL_PATH = Path("models")
MODEL_PATH.mkdir(exist_ok=True)
MODEL_NAME = "01_pytorch_workflow_model_1.pth"
MODEL_SAVE_PATH = MODEL_PATH / MODEL_NAME

print(f"Saving model to: {MODEL_SAVE_PATH}")
torch.save(obj=model_1.state_dict(), f=MODEL_SAVE_PATH)

# Load model
loaded_model_1 = LinearRegressionModelV2()
loaded_model_1.load_state_dict(torch.load(f=MODEL_SAVE_PATH))
loaded_model_1.to(device)

print(f"Loaded model structure: {loaded_model_1}")

# Verify loaded model works the same
with torch.inference_mode():
    loaded_preds = loaded_model_1(X_test.to(device))
    
print(f"\nOriginal predictions equal loaded predictions: {torch.equal(y_preds, loaded_preds)}")

# 7:37:02 Device-Agnostic Practice (data to device)
# Move data to target device
X_train = X_train.to(device)
X_test = X_test.to(device)
y_train = y_train.to(device)
y_test = y_test.to(device)

print(f"Training data on device: {X_train.device}")
print(f"Model on device: {next(model_1.parameters()).device}")